# XGBoost Reshuffled: Old Train, New Test DIFF Only

This notebook trains XGBoost on the old `preprocessed_data.csv`, saves the trained DIFF model bundle, then tests it on the new `data_clean/pre_fight_career_stats.csv` compiled into the old feature format.

This version intentionally follows the old notebook more closely:

- stance and weight-class columns are excluded
- fighter weight columns are also excluded
- old-style balancing is used: flip excess majority rows, then mirror a random equal chunk from both classes
- paired `R_*`/`B_*` features are median-filled and standardized first with a shared suffix-level scaler
- DIFF features are then built as `scaled_R - scaled_B`
- XGBoost uses 120 trees


In [9]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 43
SPLIT_SEEDS = [11, 22, 33, 44, 55]
TEST_SIZE = 0.10

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data_clean").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OLD_DATA_PATH = PROJECT_ROOT / "old" / "other dude's code" / "preprocessed_data.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data_clean" / "pre_fight_career_stats.csv"
MODEL_DIR = PROJECT_ROOT / "winner" / "models"
MODEL_DIR.mkdir(exist_ok=True)

XGB_PARAMS = dict(
    n_estimators=120,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
)


## Load And Align Datasets

In [10]:
old_df = pd.read_csv(OLD_DATA_PATH)
new_df = pd.read_csv(NEW_DATA_PATH)

old_df = old_df[old_df["Winner"].isin(["Red", "Blue"])].copy().reset_index(drop=True)
new_df = new_df[new_df["Winner"].isin(["Red", "Blue"])].copy().reset_index(drop=True)

DROP_PREFIXES = (
    "B_Stance_",
    "R_Stance_",
    "B_Stance",
    "R_Stance",
    "weight_class_",
)
DROP_EXACT = {
    "B_Weight_lbs",
    "R_Weight_lbs",
}
LEAKAGE_OR_METADATA_COLS = {
    "Winner",
    "winner",
    "winner_side",
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "win_by",
    "fight_type",
}


def keep_feature_col(col):
    if col in DROP_EXACT or col in LEAKAGE_OR_METADATA_COLS:
        return False
    return not col.startswith(DROP_PREFIXES)


old_feature_cols = [col for col in old_df.columns if keep_feature_col(col)]
new_feature_cols = [col for col in new_df.columns if keep_feature_col(col)]

missing_in_new = sorted(set(old_feature_cols) - set(new_feature_cols))
extra_in_new = sorted(set(new_feature_cols) - set(old_feature_cols))
for col in missing_in_new:
    new_df[col] = np.nan

# Strict old schema order for every experiment.
new_df = new_df[[col for col in new_df.columns if col not in old_feature_cols] + old_feature_cols]

print("Old dataset:", old_df.shape)
print("New dataset:", new_df.shape)
print("Old target distribution before balancing/reshuffling:")
print(old_df["Winner"].value_counts())
print("\nNew target distribution before balancing/reshuffling:")
print(new_df["Winner"].value_counts())
print("\nOld feature count after dropping stance, weight class, and fighter weights:", len(old_feature_cols))
print("Missing old columns added to new:", missing_in_new)
print("Extra new columns ignored:", extra_in_new)
print("Dropped exact columns:", sorted(DROP_EXACT))
print("Old majority baseline:", old_df["Winner"].value_counts(normalize=True).max().round(4))
print("New majority baseline:", new_df["Winner"].value_counts(normalize=True).max().round(4))


Old dataset: (5902, 160)
New dataset: (8549, 142)
Old target distribution before balancing/reshuffling:
Winner
Red     3979
Blue    1923
Name: count, dtype: int64

New target distribution before balancing/reshuffling:
Winner
Red     5477
Blue    3072
Name: count, dtype: int64

Old feature count after dropping stance, weight class, and fighter weights: 133
Missing old columns added to new: []
Extra new columns ignored: []
Dropped exact columns: ['B_Weight_lbs', 'R_Weight_lbs']
Old majority baseline: 0.6742
New majority baseline: 0.6407


## Old-style Balance And Reshuffle

In [11]:
def mirror_side_rows(rows, feature_cols):
    mirrored = rows.copy()
    suffixes = sorted({
        col[2:]
        for col in feature_cols
        if col.startswith("R_") and f"B_{col[2:]}" in feature_cols
    })

    for suffix in suffixes:
        r_col = f"R_{suffix}"
        b_col = f"B_{suffix}"
        r_values = mirrored[r_col].copy()
        mirrored[r_col] = mirrored[b_col].to_numpy()
        mirrored[b_col] = r_values.to_numpy()

    mirrored["Winner"] = mirrored["Winner"].map({"Red": "Blue", "Blue": "Red"})
    if "winner_side" in mirrored.columns:
        mirrored["winner_side"] = mirrored["Winner"].map({"Red": "R", "Blue": "B"})
    return mirrored


def old_balance_and_reshuffle(data, feature_cols, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    balanced = data.copy()

    # Step 1: old-style balancing. Remove excess majority rows, mirror them, add them back.
    counts = balanced["Winner"].value_counts()
    majority = counts.idxmax()
    total = counts.sum()
    target_majority = int(np.ceil(total / 2))
    n_to_flip = int(counts[majority] - target_majority)

    if n_to_flip > 0:
        flip_idx = rng.choice(balanced.index[balanced["Winner"] == majority].to_numpy(), size=n_to_flip, replace=False)
        flip_rows = balanced.loc[flip_idx]
        balanced = balanced.drop(index=flip_idx)
        balanced = pd.concat([balanced, mirror_side_rows(flip_rows, feature_cols)], axis=0).sort_index()

    after_balance_counts = balanced["Winner"].value_counts()

    # Step 2: old-style reshuffle. Mirror an equal random chunk from both classes.
    sample_n = int(after_balance_counts.min() // 2)
    red_idx = rng.choice(balanced.index[balanced["Winner"] == "Red"].to_numpy(), size=sample_n, replace=False)
    blue_idx = rng.choice(balanced.index[balanced["Winner"] == "Blue"].to_numpy(), size=sample_n, replace=False)
    reshuffle_idx = np.concatenate([red_idx, blue_idx])

    reshuffle_rows = balanced.loc[reshuffle_idx]
    balanced = balanced.drop(index=reshuffle_idx)
    balanced = pd.concat([balanced, mirror_side_rows(reshuffle_rows, feature_cols)], axis=0).sort_index().reset_index(drop=True)

    balanced["was_old_balance_or_reshuffle_swapped"] = False
    balanced.loc[balanced.index.isin([]), "was_old_balance_or_reshuffle_swapped"] = False

    return balanced, {
        "balance_flipped_rows": n_to_flip,
        "reshuffled_rows_per_side": sample_n,
        "reshuffled_rows_total": sample_n * 2,
    }


old_balanced_df, old_balance_info = old_balance_and_reshuffle(old_df, old_feature_cols, seed=RANDOM_STATE)
new_balanced_df, new_balance_info = old_balance_and_reshuffle(new_df, old_feature_cols, seed=RANDOM_STATE)

print("Old target distribution after old-style balance/reshuffle:")
print(old_balanced_df["Winner"].value_counts())
print(old_balance_info)
print("\nNew target distribution after old-style balance/reshuffle:")
print(new_balanced_df["Winner"].value_counts())
print(new_balance_info)


Old target distribution after old-style balance/reshuffle:
Winner
Red     2951
Blue    2951
Name: count, dtype: int64
{'balance_flipped_rows': 1028, 'reshuffled_rows_per_side': 1475, 'reshuffled_rows_total': 2950}

New target distribution after old-style balance/reshuffle:
Winner
Red     4275
Blue    4274
Name: count, dtype: int64
{'balance_flipped_rows': 1202, 'reshuffled_rows_per_side': 2137, 'reshuffled_rows_total': 4274}


C:\Users\Abtin\AppData\Local\Temp\ipykernel_7816\433937249.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  balanced["was_old_balance_or_reshuffle_swapped"] = False
C:\Users\Abtin\AppData\Local\Temp\ipykernel_7816\433937249.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  balanced["was_old_balance_or_reshuffle_swapped"] = False


## Shared-scale R/B Features, Then Build DIFF


In [12]:
def make_y(data):
    return data["Winner"].map({"Blue": 0, "Red": 1}).astype(int)


def fit_shared_scaled_diff(train_data, test_data, feature_cols):
    """
    Fit preprocessing on the old training data, then apply it to both old and new.

    For each paired suffix, R_suffix and B_suffix share one median/mean/std learned from
    the combined old R and B values. DIFF is then scaled_R - scaled_B.
    """
    train_out = pd.DataFrame(index=train_data.index)
    test_out = pd.DataFrame(index=test_data.index)
    preprocessors = {}

    context_cols = [col for col in feature_cols if not col.startswith(("R_", "B_"))]
    if context_cols:
        context_imputer = SimpleImputer(strategy="median")
        context_scaler = StandardScaler()

        train_context = train_data[context_cols].apply(pd.to_numeric, errors="coerce")
        test_context = test_data[context_cols].apply(pd.to_numeric, errors="coerce")

        train_context_imputed = context_imputer.fit_transform(train_context)
        train_out[context_cols] = context_scaler.fit_transform(train_context_imputed)
        test_out[context_cols] = context_scaler.transform(context_imputer.transform(test_context))

        preprocessors["context"] = {
            "columns": context_cols,
            "imputer": context_imputer,
            "scaler": context_scaler,
        }

    suffixes = sorted({
        col[2:]
        for col in feature_cols
        if col.startswith("R_") and f"B_{col[2:]}" in feature_cols
    })

    for suffix in suffixes:
        r_col = f"R_{suffix}"
        b_col = f"B_{suffix}"

        train_r = pd.to_numeric(train_data[r_col], errors="coerce")
        train_b = pd.to_numeric(train_data[b_col], errors="coerce")
        train_combined = pd.concat([train_r, train_b], ignore_index=True)

        median = train_combined.median()
        if pd.isna(median):
            median = 0.0
        train_combined_filled = train_combined.fillna(median)
        mean = train_combined_filled.mean()
        std = train_combined_filled.std(ddof=0)
        if pd.isna(std) or std == 0:
            std = 1.0

        train_r_scaled = (train_r.fillna(median) - mean) / std
        train_b_scaled = (train_b.fillna(median) - mean) / std
        test_r_scaled = (pd.to_numeric(test_data[r_col], errors="coerce").fillna(median) - mean) / std
        test_b_scaled = (pd.to_numeric(test_data[b_col], errors="coerce").fillna(median) - mean) / std

        diff_col = f"DIFF_{suffix}"
        train_out[diff_col] = train_r_scaled - train_b_scaled
        test_out[diff_col] = test_r_scaled - test_b_scaled
        preprocessors[suffix] = {
            "r_col": r_col,
            "b_col": b_col,
            "median": median,
            "mean": mean,
            "std": std,
        }

    return train_out, test_out, preprocessors


X_old_diff, X_new_diff, diff_preprocessors = fit_shared_scaled_diff(
    old_balanced_df,
    new_balanced_df,
    old_feature_cols,
)
y_old = make_y(old_balanced_df)
y_new = make_y(new_balanced_df)

print("Shared-scaled DIFF train/test shapes:", X_old_diff.shape, X_new_diff.shape)
print("Old balanced y:")
print(y_old.value_counts())
print("New balanced y:")
print(y_new.value_counts())
print("Feature mean sample:")
print(X_old_diff.mean().head().round(6))
X_old_diff.head()


Shared-scaled DIFF train/test shapes: (5902, 67) (8549, 67)
Old balanced y:
Winner
1    2951
0    2951
Name: count, dtype: int64
New balanced y:
Winner
1    4275
0    4274
Name: count, dtype: int64
Feature mean sample:
title_bout           0.000000
DIFF_Height_cms     -0.005940
DIFF_Reach_cms      -0.015191
DIFF_age             0.008594
DIFF_avg_BODY_att    0.003037
dtype: float64


,title_bout,DIFF_Height_cms,DIFF_Reach_cms,DIFF_age,DIFF_avg_BODY_att,DIFF_avg_BODY_landed,DIFF_avg_CLINCH_att,DIFF_avg_CLINCH_landed,DIFF_avg_CTRL_time(seconds),DIFF_avg_DISTANCE_att,...,DIFF_total_rounds_fought,DIFF_total_time_fought(seconds),DIFF_total_title_bouts,DIFF_win_by_Decision_Majority,DIFF_win_by_Decision_Split,DIFF_win_by_Decision_Unanimous,DIFF_win_by_KO/TKO,DIFF_win_by_Submission,DIFF_win_by_TKO_Doctor_Stoppage,DIFF_wins
0,-0.25487,0.579457,0.740957,-0.980124,-0.712042,-0.920874,-1.305274,-0.663591,-0.233920,-0.044684,...,-0.226545,-1.521623,0.000000,0.0,0.0,0.000000,0.577561,-0.78938,0.0,0.000000
1,-0.25487,-0.579457,-0.493972,-0.980124,-0.946692,-0.863319,0.888789,0.598417,-0.349620,0.006982,...,0.981694,0.707081,0.000000,0.0,0.0,0.000000,1.155122,0.00000,0.0,0.549453
2,-0.25487,0.000000,0.000000,0.980124,0.222513,0.097539,-0.766419,-0.912438,0.135180,0.062837,...,-0.981694,0.792230,0.000000,0.0,0.0,-0.646497,-1.732683,0.00000,0.0,-1.098905
3,-0.25487,-0.869186,-0.493972,0.735093,-0.054111,-0.065227,0.000000,0.000000,-0.045601,-0.052233,...,0.000000,-0.072206,0.000000,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.000000
4,-0.25487,1.448643,0.987943,-3.430433,1.801472,2.342293,2.757069,2.813968,2.008206,-1.168115,...,-1.434784,-0.459530,0.736227,0.0,0.0,0.000000,0.000000,-0.78938,0.0,-0.274726


## Shared-scaled Feature Alignment Check


In [13]:
assert X_old_diff.shape[1] == X_new_diff.shape[1]
assert list(X_old_diff.columns) == list(X_new_diff.columns)
assert X_old_diff.isna().sum().sum() == 0
assert X_new_diff.isna().sum().sum() == 0

print("Shared-scaled DIFF features are aligned and contain no missing values.")
print("Feature count:", X_old_diff.shape[1])


Shared-scaled DIFF features are aligned and contain no missing values.
Feature count: 67


## Old-data 5x 90/10 Sanity Check On Shared-scaled DIFF Data


In [14]:
def make_xgb_model(seed):
    return XGBClassifier(random_state=seed, **XGB_PARAMS)


def evaluate_splits(name, X, y):
    rows = []
    for split_num, seed in enumerate(SPLIT_SEEDS, start=1):
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        train_idx, test_idx = next(splitter.split(X, y))
        model = make_xgb_model(seed)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[test_idx])
        proba = model.predict_proba(X.iloc[test_idx])[:, 1]
        rows.append({
            "feature_set": name,
            "split": split_num,
            "seed": seed,
            "accuracy": accuracy_score(y.iloc[test_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y.iloc[test_idx], pred),
            "roc_auc": roc_auc_score(y.iloc[test_idx], proba),
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
        })
    return pd.DataFrame(rows)


old_diff_splits = evaluate_splits("Old-style balanced/reshuffled shared-scaled DIFF", X_old_diff, y_old)
old_split_summary = (
    old_diff_splits
    .groupby("feature_set")
    .agg(
        n_splits=("split", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
    )
    .reset_index()
    .sort_values("mean_accuracy", ascending=False)
)

display(old_split_summary)
display(old_diff_splits[["split", "seed", "accuracy", "balanced_accuracy", "roc_auc", "train_rows", "test_rows"]])


,feature_set,n_splits,mean_accuracy,std_accuracy,mean_balanced_accuracy,mean_roc_auc,std_roc_auc
0,Old-style balanced/reshuffled shared-scaled DIFF,5,0.634856,0.010992,0.634866,0.685544,0.01299


,split,seed,accuracy,balanced_accuracy,roc_auc,train_rows,test_rows
0,1,11,0.631134,0.631167,0.667230,5311,591
1,2,22,0.620981,0.620969,0.678373,5311,591
2,3,33,0.636210,0.636177,0.699800,5311,591
3,4,44,0.634518,0.634563,0.694463,5311,591
4,5,55,0.651438,0.651454,0.687855,5311,591


## Train On Full Old Shared-scaled DIFF Dataset, Save Model, Test On Full New Shared-scaled DIFF Dataset


In [15]:
def fit_save_and_test(name, X_train, y_train, X_test, y_test, model_path):
    model = make_xgb_model(RANDOM_STATE)
    model.fit(X_train, y_train)
    joblib.dump({
        "model": model,
        "shared_diff_preprocessors": diff_preprocessors,
        "feature_columns": X_train.columns.tolist(),
        "target_mapping": {"Blue": 0, "Red": 1},
        "xgb_params": XGB_PARAMS,
        "old_style_balanced_and_reshuffled": True,
        "shared_scaled_before_diff": True,
        "dropped_columns": sorted(DROP_EXACT),
        "feature_set": "DIFF",
    }, model_path)

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    result = {
        "feature_set": name,
        "train_rows_old": len(X_train),
        "test_rows_new": len(X_test),
        "n_features": X_train.shape[1],
        "accuracy_on_new": accuracy_score(y_test, pred),
        "balanced_accuracy_on_new": balanced_accuracy_score(y_test, pred),
        "roc_auc_on_new": roc_auc_score(y_test, proba),
        "saved_model": str(model_path),
    }
    return result, pred, proba, model


diff_result, diff_pred, diff_proba, diff_model = fit_save_and_test(
    "Old-style balanced/reshuffled shared-scaled DIFF -> new shared-scaled DIFF",
    X_old_diff,
    y_old,
    X_new_diff,
    y_new,
    MODEL_DIR / "xgb_old_diff_schema_old_reshuffle_shared_scaled_120.joblib",
)

cross_dataset_results = pd.DataFrame([diff_result])
display(cross_dataset_results)


,feature_set,train_rows_old,test_rows_new,n_features,accuracy_on_new,balanced_accuracy_on_new,roc_auc_on_new,saved_model
0,Old-style balanced/reshuffled shared-scaled DI...,5902,8549,67,0.634928,0.634928,0.687734,c:\Users\Abtin\Documents\Github\UFC-Prediction...


## New-dataset DIFF Diagnostics

In [16]:
def show_new_test_diagnostics(name, y_true, pred, proba, feature_names, fitted_model):
    print(f"\n{name}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, pred))
    print(classification_report(y_true, pred, target_names=["Blue", "Red"]))
    print("Predicted label counts:")
    print(pd.Series(pred).map({0: "Blue", 1: "Red"}).value_counts())

    if hasattr(fitted_model, "feature_importances_"):
        importance = pd.Series(fitted_model.feature_importances_, index=feature_names).sort_values(ascending=False)
        display(importance.head(30).to_frame("importance"))


show_new_test_diagnostics(
    "Old-style balanced/reshuffled shared-scaled DIFF model tested on full new dataset",
    y_new,
    diff_pred,
    diff_proba,
    X_old_diff.columns,
    diff_model,
)



Old-style balanced/reshuffled shared-scaled DIFF model tested on full new dataset
Confusion matrix:
[[2714 1560]
 [1561 2714]]
              precision    recall  f1-score   support

        Blue       0.63      0.64      0.63      4274
         Red       0.64      0.63      0.63      4275

    accuracy                           0.63      8549
   macro avg       0.63      0.63      0.63      8549
weighted avg       0.63      0.63      0.63      8549

Predicted label counts:
Blue    4275
Red     4274
Name: count, dtype: int64


,importance
DIFF_longest_win_streak,0.033880
DIFF_avg_GROUND_landed,0.033249
DIFF_age,0.027579
DIFF_current_win_streak,0.025827
DIFF_avg_TD_landed,0.024584
DIFF_avg_opp_DISTANCE_landed,0.024156
DIFF_avg_opp_TOTAL_STR_att,0.023794
DIFF_avg_KD,0.022777
DIFF_Reach_cms,0.022762
DIFF_avg_opp_SIG_STR_pct,0.021682
